# ONNX Inference Smoke Test

Run a one-batch ONNX Runtime sanity check against the exported deployment model.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import onnxruntime as ort

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())


In [ ]:
def smoke_test_onnx(onnx_path: Path, batch: np.ndarray) -> dict[str, object]:
    session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    outputs = session.run(None, {input_name: batch.astype(np.float32)})
    return {
        'input_name': input_name,
        'output_shapes': [list(output.shape) for output in outputs],
    }


In [ ]:
ONNX_PATH = Path(str(override('ONNX_PATH', TRAINING_OUTPUTS_ROOT / 'mobilenetv3small_8samples_final_deployment_cnn_only' / 'models' / 'meatlens_final_8samples_cnn_only_mobilenetv3small.onnx')))
SMOKE_SUMMARY_PATH = Path(str(override('SMOKE_SUMMARY_PATH', ONNX_PATH.with_suffix('.smoke.json'))))
batch = np.zeros((1, INPUT_SHAPE[0], INPUT_SHAPE[1], INPUT_SHAPE[2]), dtype=np.float32)
summary = smoke_test_onnx(ONNX_PATH, batch)
summary['onnx_path'] = str(ONNX_PATH)
summary['expected_input_shape'] = list(INPUT_SHAPE)
SMOKE_SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
